# Campaign finance data processing 

Every election cycle has been different, because the way SOS has reported this data has changed and the types of races and things we've covered have changed. 

This year, I used https://campaignfinance.vermont.gov/public/cf/contribution to download the data and used the Datawrapper API to automatically update the charts for the Election Guide. Note that I have pre-filtered the data to just include statewide races; the full dataset proved too unwieldy. 

In [1]:
import pandas as pd
from datawrapper import Datawrapper

/Users/erinpetenko/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
# this is the "password" of sorts for accessing the Datawrapper API to create charts in Python
DATAWRAPPER_ACCESS_TOKEN = "KEY-REDACTED"


### Part 1: Candidate-by-candidate chart

In [3]:
# this is reading the csv into Pandas for further processing. The data automatically downloads with a stupid header line, so I told Pandas to skip a row
df = pd.read_csv("Contributions_10_15_2026.csv", skiprows=1)

In [4]:
# a quick check that it loaded ok.
df.head()

,Entity Id,Committee Name,Candidate First Name,Candidate Last Name,Candidate Middle Name,Contributor Type,Contributor Name,Address Line1,Address Line2,City,State Code,Zip Code,Transaction Date,Transaction Amount,Election Cycle,Filer Type,Contribution Type,Timed Report,Report Name
0,183319,NaN,MOLLY,GRAY,NaN,Individual,"COOKE, RICHARD",37 MAPLE STREET,NaN,BRATTLEBORO,VT,5301,10/12/2026,$50.00,2026,Candidate,Contribution,NaN,2026 Oct 15 - Disclosures
1,301161,NaN,TIM,ASHE,NaN,Individual,"ASHE, CHRISTOPHER",711 N IRVING STREET,NaN,ARLINGTON,VA,22201,10/12/2026,$250.00,2026,Candidate,Contribution,NaN,2026 Oct 15 - Disclosures
2,183319,NaN,MOLLY,GRAY,NaN,Individual,"CASSON, PETER",139 RABBITS RUN,NaN,WILLISTON,VT,5495,10/12/2026,$500.00,2026,Candidate,Contribution,NaN,2026 Oct 15 - Disclosures
3,183319,NaN,MOLLY,GRAY,NaN,Individual,"RIDER, ANNE",2676 TATER LN,NaN,GUILFORD,VT,5301,10/12/2026,$250.00,2026,Candidate,Contribution,NaN,2026 Oct 15 - Disclosures
4,183319,NaN,MOLLY,GRAY,NaN,NaN,Under Threshold - Name Withheld,NaN,NaN,NaN,VT,NaN,10/12/2026,$100.00,2026,Candidate,Contribution,NaN,2026 Oct 15 - Disclosures


In [5]:
df['Contribution Type'].value_counts()

Contribution Type
Contribution    19402
Name: count, dtype: int64

In [6]:
# I noticed the Transaction Amount came with $ symbols which made pandas interpret it as a string. 
# so I removed those symbols and then converted the column into a numeric one.
df['Transaction Amount'] = df['Transaction Amount'].str.replace("$",repl="")
df['Transaction Amount'] = df['Transaction Amount'].str.replace(",",repl="")

df['Transaction Amount'] = pd.to_numeric(df['Transaction Amount'],errors='raise')

In [7]:
# a spare column just so that I could see which candidates were in the data
df['Candidate Full Name'] = df['Candidate First Name'] + " " + df['Candidate Last Name']

In [8]:
df['Candidate Full Name'].unique()

array(['MOLLY GRAY', 'TIM ASHE', 'AMANDA JANOO', 'JOHN RODGERS',
       'PHIL SCOTT', 'LYNN LAFLEUR', 'MIKE PIECIAK',
       'SARAH COPELAND HANZAS', 'CHARITY CLARK', 'BRYAN PONNWITZ',
       'EDWIN KEMON', 'DAVID ZUCKERMAN', 'ALY RICHARDS', 'BRIAN JUDD',
       'CHARLES KRONICK', 'RYAN MCLAREN', 'ZACHARY HAMPL',
       'ESTHER CHARLESTIN', 'JIM CONDOS', 'DANIEL TOWLE',
       'NICHOLAS GRAETER', 'JUNE GOODBAND', 'RACHEL SHAW',
       'THOMAS RENNER'], dtype=object)

In [9]:
#this is a reference spreadsheet with the standardized candidate names, offices, and the candidate ID 
canddict = pd.read_csv("https://docs.google.com/spreadsheets/d/e/2PACX-1vSXcK5v0Wrfi7FLkdJFxPx55rLGZPOkThbxOfjmS7QxEgk5JV_O1QSaIQ5g_AnR9q6aTMlng_Olu41R/pub?gid=0&single=true&output=csv",
    index_col="Entity Id")

In [10]:
# now I'm aggregating the data by amount and number of contributions into a pivot table
candpiv = df.pivot_table(index="Entity Id",values=['Transaction Amount'],
               aggfunc="sum").merge(df.pivot_table(index="Entity Id", 
                                                 values='Transaction Date',aggfunc=len),left_index=True,
                                  right_index=True).rename(columns={"Transaction Amount": "Amount raised",
    "Transaction Date":"Number of contributions"}).sort_values(by="Amount raised",ascending=False)

candpiv.head(10)

,Amount raised,Number of contributions
Entity Id,,
301221,898555.83,7010
33545,588881.57,1153
183319,562864.28,2503
301230,482829.68,1986
301167,428545.34,2183
285236,368804.53,676
242612,257389.38,2367
301017,179589.50,486
301161,76938.31,514


In [11]:
# merging the pivot table with the candidate dictionary
candmerge = canddict.merge(candpiv, left_index=True,right_index=True)
candmerge

,Candidate Full Name,VTDigger name,In SOS spreadsheet?,Office,In general?,Amount raised,Number of contributions
Entity Id,,,,,,,
301017,CHARITY CLARK,Charity R. Clark,Yes,Attorney General,Yes,179589.50,486
301161,TIM ASHE,Tim Ashe,Yes,Auditor of Accounts,Yes,76938.31,514
301502,CHARLES KRONICK,Ivar Kronick,Yes,Auditor of Accounts,Yes,6670.00,6
301221,AMANDA JANOO,Amanda Janoo,Yes,Governor,Yes,898555.83,7010
33545,PHIL SCOTT,Phil Scott,Yes,Governor,Yes,588881.57,1153
301494,BRYAN PONNWITZ,Bryan Ponnwitz,No,Governor,Yes,4964.67,17
285236,JOHN RODGERS,John S. Rodgers,Yes,Lieutenant Governor,Yes,368804.53,676
183319,MOLLY GRAY,Molly Gray,Yes,Lieutenant Governor,Yes,562864.28,2503
301332,RACHEL SHAW,Rachel Shaw,Yes,Secretary of State,Yes,100.00,1


In [12]:
# making some tweaks for what it will look like to display it in Datawrapper
canddisplay = candmerge[candmerge['In SOS spreadsheet?'] == "Yes"].rename(columns={"VTDigger name":"Candidate"})[["Candidate", "Office",  "Amount raised","Number of contributions"]]

canddisplay

# canddisplay.to_clipboard()

,Candidate,Office,Amount raised,Number of contributions
Entity Id,,,,
301017,Charity R. Clark,Attorney General,179589.50,486
301161,Tim Ashe,Auditor of Accounts,76938.31,514
301502,Ivar Kronick,Auditor of Accounts,6670.00,6
301221,Amanda Janoo,Governor,898555.83,7010
33545,Phil Scott,Governor,588881.57,1153
285236,John S. Rodgers,Lieutenant Governor,368804.53,676
183319,Molly Gray,Lieutenant Governor,562864.28,2503
301332,Rachel Shaw,Secretary of State,100.00,1
242595,Sarah Copeland Hanzas,Secretary of State,20499.06,159


In [13]:
dw = Datawrapper(DATAWRAPPER_ACCESS_TOKEN)

#ID for the candidate chart I created
cand_chart = 'dtLeu'

# Updating the data in the chart
dw.add_data(
 cand_chart,
 canddisplay
)


True

### Part 2: Donor Types

In [14]:
#Cleaning donor type data
df['Contributor Type'].value_counts()

Contributor Type
Individual                     7918
Business/Group/Organization     199
Candidate                        25
Self                             24
Political Party                  16
PAC                               9
Name: count, dtype: int64

In [15]:
df['Contributor Type Cleaned'] = df.apply(lambda x: "Total of contributions $100.00 or less" if x['Contributor Name'] == 'Under Threshold - Name Withheld' else x['Contributor Type'], axis=1)

df['Contributor Type Cleaned'].value_counts()

Contributor Type Cleaned
Total of contributions $100.00 or less    11211
Individual                                 7918
Business/Group/Organization                 199
Candidate                                    25
Self                                         24
Political Party                              16
PAC                                           9
Name: count, dtype: int64

In [16]:
catpiv = df.pivot_table(index='Contributor Type Cleaned', values='Transaction Amount', aggfunc='sum').sort_values(by='Transaction Amount', ascending=False)

catpiv

,Transaction Amount
Contributor Type Cleaned,
Individual,3083648.41
Total of contributions $100.00 or less,465863.04
Business/Group/Organization,267923.76
Political Party,61678.80
PAC,23890.00
Candidate,22473.00
Self,16228.34


In [17]:
# creating a pivot table of the cleaned data
catdisplay = catpiv.reset_index().rename(columns={'Contributor Type Cleaned': 'Contributor Type',
                                                  'Transaction Amount':'Amount'})

In [18]:
catdisplay

,Contributor Type,Amount
0,Individual,3083648.41
1,Total of contributions $100.00 or less,465863.04
2,Business/Group/Organization,267923.76
3,Political Party,61678.80
4,PAC,23890.00
5,Candidate,22473.00
6,Self,16228.34


In [19]:
#ID for the candidate chart I created
cat_chart = 'udRX9'

# Updating the data in the chart
dw.add_data(
 cat_chart,
 catdisplay
)


True

### Part 3: Historic data comparison

In [20]:

# checking which reports have been submitted so far this election cycle
df['Report Name'].unique()

array(['2026 Oct 15 - Disclosures', '2026 Oct 1 - Disclosures',
       '2026 Sep 1 - Disclosures', '2026 Aug 1 - Disclosures',
       '2026 Jul 1 - Disclosures', '2026 Mar 15 - Disclosures',
       '2025 Jul 1 - Disclosures'], dtype=object)

In [21]:
# Reading in the 2024 data 

df_2024 = pd.read_csv("archived_data/2024_contributions.csv", skiprows=1)

df_2024['Transaction Amount'] = df_2024['Transaction Amount'].str.replace("$",repl="")
df_2024['Transaction Amount'] = df_2024['Transaction Amount'].str.replace(",",repl="")

df_2024['Transaction Amount'] = pd.to_numeric(df_2024['Transaction Amount'],errors='raise')



In [22]:
df_2024.head()

,Entity Id,Committee Name,Candidate First Name,Candidate Last Name,Candidate Middle Name,Contributor Type,Contributor Name,Address Line1,Address Line2,City,State Code,Zip Code,Transaction Date,Transaction Amount,Election Cycle,Filer Type,Contribution Type,Timed Report,Report Name
0,227923,NaN,DAVID,ZUCKERMAN,NaN,Political Party,VERMONT PROGRESSIVE PARTY,PO BOX 281,NaN,MONTPELIER,VT,05601,12/12/2024,5857.47,2024,Candidate,Contribution,NaN,12/15/2024 - GENERAL
1,285236,NaN,JOHN,RODGERS,NaN,Individual,"POMERLEAU, ERNEST",69 COLLEGE STREET,NaN,BURLINGTON,VT,05401,12/12/2024,2000.00,2024,Candidate,Contribution,NaN,12/15/2024 - GENERAL
2,242612,NaN,MIKE,PIECIAK,NaN,Individual,"D'ALESSANDRO, MAC",333 BRUSH HILL RD,UNIT Z3,MILTON,MA,02186-1243,12/12/2024,250.00,2024,Candidate,Contribution,NaN,12/15/2024 - GENERAL
3,227923,NaN,DAVID,ZUCKERMAN,NaN,NaN,Under Threshold - Name Withheld,NaN,NaN,NaN,NaN,NaN,12/11/2024,10.00,2024,Candidate,Contribution,NaN,12/15/2024 - GENERAL
4,227923,NaN,DAVID,ZUCKERMAN,NaN,Individual,"ANDREWS, CATHARINE H.",235 PARK ST,#1,BURLINGTON,VT,05401,12/11/2024,15.00,2024,Candidate,Contribution,NaN,12/15/2024 - GENERAL


In [23]:

# checking which reports are in the 2024 data
df_2024['Report Name'].unique()

array(['12/15/2024 - GENERAL', '11/19/2024 - GENERAL',
       '11/01/2024 - GENERAL', '10/15/2024 - GENERAL',
       '10/01/2024 - GENERAL', '09/01/2024 - GENERAL',
       '08/01/2024 - GENERAL', '07/01/2024 - GENERAL',
       '03/15/2024 - GENERAL', '07/01/2023 - GENERAL'], dtype=object)

In [24]:
# filtering down to just the 2024 reports that match up with the 2026 ones. This isn't always perfect, because the dates 
# have occasionally changed over the years, but I do my best.

reports_to_keep = ['07/01/2024 - GENERAL', 
       '03/15/2024 - GENERAL', '07/01/2023 - GENERAL', '08/01/2024 - GENERAL',
                '09/01/2024 - GENERAL' , '10/15/2024 - GENERAL',
       '10/01/2024 - GENERAL']

df_2024_filtered = df_2024[df_2024['Report Name'].isin(reports_to_keep)]

In [25]:
# checking that the numbers coming out make sense & formatting them correctly

print('${:,.0f}'.format(df['Transaction Amount'].sum()))
print('${:,.0f}'.format(df_2024_filtered['Transaction Amount'].sum()))

$3,941,705
$1,260,278


In [26]:
comp_chart = 'Vmtpi'

# Updating the data in the chart
dw.update_description(
 chart_id = comp_chart,
 intro = "<b>" + '${:,.0f}'.format(df['Transaction Amount'].sum()) + "</b> has been raised so far by statewide candidates in the 2026 election cycle.<br><br>That's compared to <b>" + '${:,.0f}'.format(df_2024_filtered['Transaction Amount'].sum()) + "</b> raised at about this point in the 2024 cycle.<br><br><b>Statewide candidate contributions for previous years:</b>"
)


{'publicId': 'Vmtpi',
 'language': 'en-US',
 'theme': 'datawrapper',
 'id': 'Vmtpi',
 'type': 'column-chart',
 'title': 'Contributions by year 2026',
 'lastEditStep': 5,
 'publishedAt': '2026-07-27T20:46:18.000Z',
 'publicUrl': 'https://datawrapper.dwcdn.net/Vmtpi/6/',
 'publicVersion': 6,
 'deleted': False,
 'forkable': False,
 'isFork': False,
 'metadata': {'data': {'changes': [],
   'transpose': False,
   'vertical-header': True,
   'horizontal-header': True,
   'column-format': {'Total with inflation': {'type': 'auto',
     'ignore': False,
     'number-append': '',
     'number-format': 'auto',
     'number-divisor': 0,
     'number-prepend': '$'}},
   'upload-method': 'copy'},
  'describe': {'source-name': 'Vermont Secretary of State',
   'source-url': 'https://campaignfinance.vermont.gov/Public/ReceiptsList',
   'intro': "<b>$3,941,705</b> has been raised so far by statewide candidates in the 2026 election cycle.<br><br>That's compared to <b>$1,260,278</b> raised at about this p